# 03 — Medallion ELT Benchmark on BigQuery DataFrames (`bigframes.pandas`, Serverless BigQuery)

This notebook executes the full **TPC-DS Omnichannel Medallion pipeline** (`SPEC.md` §3) using the official **BigQuery DataFrames (`bigframes.pandas >= 2.40`)** Python API compiled lazily to BigQuery GoogleSQL, executed inside a non-bypassable Enterprise slot reservation (`BQ_SLOT_CAP` slots in `US`).

### Benchmark Design & Architectural Controls (Passes 4–7)
1. **Partial Ordering Mode (`ordering_mode = "partial"`):** Explicitly configured before session creation so `BigFrames` pushes down distributed unordered relational operators directly to BigQuery slots without generating sequential whole-table row-hash ordering indexes.
2. **Storage & Materialization Pattern Passes (4 Variants via `BF_PASS_FILTER`):**
   - **Pass 4 (`bigframes_biglake_gcs` / `biglake_gcs_parquet`):** Reads all 8 Bronze tables from `${PROJECT_ID}.${DS_BRONZE_BIGLAKE}` (BigLake external tables backed by the exact same GCS Parquet files read by Dataproc Spark `01`/`02`) and writes via `df.to_gbq(..., if_exists="replace")`.
   - **Pass 5 (`bigframes_native_bq` / `bq_native_capacitor`):** Reads all 8 Bronze tables from `${PROJECT_ID}.${DS_BRONZE_NATIVE}` (native BigQuery Capacitor tables) and materializes unclustered Silver/Gold tables via `df.to_gbq(..., if_exists="replace")` (baseline highlighting `_SESSION.bqdf_*` + `MERGE` double-write overhead).
   - **Pass 6 (`bigframes_ctas_native_bq` / `bq_native_capacitor`):** Same 4-stage Medallion checkpoints (`_ctas_native` suffix) over `DS_BRONZE_NATIVE`, compiling via `df.sql` into single-pass `CREATE OR REPLACE TABLE ... CLUSTER BY ... AS SELECT ...` (`sold_date_sk, customer_sk, item_sk` for Silver; `d_date_sk, channel, i_category` for Gold) and omitting `daily.cache()` in Stage 3 so CTE/window compilation completes in a single Dremel query job.
   - **Pass 7 (`bigframes_lazy_e2e_bq` / `bq_native_capacitor`, `stage_id="stage_e2e_bronze_to_gold"`):** Chains Stage 1 -> Stage 2A -> Stage 3 lazily as one `BigFrames` expression over `DS_BRONZE_NATIVE` (zero intermediate Silver disk writes, zero `daily.cache()`) and materializes `${DS_GOLD_BQ}.gold_omnichannel_kpi_mart_lazy_e2e` in a single clustered `df.sql` CTAS job.
3. **Selective Execution Toggle (`BF_PASS_FILTER`):** Defaults to `"all"`, or accepts a comma-separated subset of `{"biglake", "native", "ctas_native", "lazy_e2e"}` (e.g. `BF_PASS_FILTER="ctas_native,lazy_e2e"`) for targeted backfills.
4. **Strict Slot Governance Verification:** Confirms `COUNTIF(reservation_id IS NULL OR STRPOS(reservation_id, @reservation_name) = 0) == 0` across every stage job (`ALLOW_UNRESERVED_BQ=False`), and tests `SET @@reservation = 'none'; SELECT 1;` truthfully.
5. **Stage 2B Python Execution Contrast (`SQL Pushdown` vs. `@bpd.udf`):** Compiles modular Python scoring rules natively into Dremel GoogleSQL pushdown inside the flat-rate reservation by default (`RUN_BQ_MANAGED_UDF="false"`), with optional `@bpd.udf` BigQuery Managed Python UDF toggle.
6. **Clean Session Lifecycle:** Wraps execution in `try ... finally: bpd.close_session()` so all session-scoped `_anonymous_*` artifacts are purged immediately.

In [1]:
import datetime
import json
import os
import re
import sys
import time
import uuid
from pathlib import Path
from typing import Any

import bigframes.pandas as bpd
import google.api_core.exceptions as gapi_exceptions
import pandas as pd
from dotenv import load_dotenv
from google.cloud import bigquery

try:
    from IPython.display import display
except ImportError:  # Fallback when running extracted code outside Jupyter/IPython.
    display = print


def resolve_repo_root(start: Path | None = None) -> Path:
    """Walk upward until SPEC.md and pyproject.toml are located."""
    cur = (start or Path.cwd()).resolve()
    for candidate in (cur, *cur.parents):
        if (candidate / "SPEC.md").is_file() and (candidate / "pyproject.toml").is_file():
            return candidate
    raise FileNotFoundError(f"Could not locate repository root above {cur}")


def _parse_bool_env(name: str, default: bool) -> bool:
    raw = os.getenv(name)
    if raw is None or not raw.strip():
        return default
    return raw.strip().lower() in {"1", "true", "yes", "on"}


def _sanitize_bq_label(value: str, max_len: int = 63) -> str:
    cleaned = re.sub(r"[^a-z0-9_-]", "-", value.strip().lower())
    return (cleaned[:max_len].strip("-_")) or "na"


VALID_BF_PASS_KEYS: tuple[str, ...] = ("biglake", "native", "ctas_native", "lazy_e2e")


def parse_bf_pass_filter(raw_filter: str | None) -> set[str]:
    """Parse BF_PASS_FILTER ('all' or comma-separated subset of VALID_BF_PASS_KEYS)."""
    cleaned = (raw_filter or "all").strip().lower()
    if not cleaned or cleaned == "all":
        return set(VALID_BF_PASS_KEYS)
    selected = {tok.strip() for tok in cleaned.split(",") if tok.strip()}
    unknown = selected - set(VALID_BF_PASS_KEYS)
    if not selected or unknown:
        raise ValueError(
            f"Invalid BF_PASS_FILTER={raw_filter!r}; expected 'all' or subset of {VALID_BF_PASS_KEYS}"
        )
    return selected


REPO_ROOT = resolve_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

load_dotenv(REPO_ROOT / ".env", override=False)

from scripts.metrics_helper import load_benchmark_runs, log_stage_metric

PROJECT_ID = os.getenv("PROJECT_ID", "").strip()
if not PROJECT_ID or PROJECT_ID == "your-gcp-project-id":
    raise ValueError(
        "PROJECT_ID must be set to a valid GCP project ID in .env or environment (not 'your-gcp-project-id')."
    )

BQ_LOCATION = os.getenv("BQ_LOCATION", "US").strip()
if not re.fullmatch(r"[A-Za-z0-9_-]+", BQ_LOCATION):
    raise ValueError(f"Invalid BQ_LOCATION: {BQ_LOCATION!r}")
BQ_INFO_SCHEMA_REGION = f"region-{BQ_LOCATION.lower()}"

BQ_ADMIN_PROJECT_ID = os.getenv("BQ_ADMIN_PROJECT_ID", PROJECT_ID).strip() or PROJECT_ID
BQ_RESERVATION_NAME = os.getenv("BQ_RESERVATION_NAME", "bench-elt-slots").strip()
BQ_SLOT_CAP = int(os.getenv("BQ_SLOT_CAP", "100"))
ALLOW_UNRESERVED_BQ = _parse_bool_env("ALLOW_UNRESERVED_BQ", False)
BIGLAKE_CONNECTION_ID = os.getenv("BIGLAKE_CONNECTION_ID", "bench-biglake-us").strip()

DS_BRONZE_BIGLAKE = os.getenv("DS_BRONZE_BIGLAKE", "bench_bronze_biglake").strip()
DS_BRONZE_NATIVE = os.getenv("DS_BRONZE_NATIVE", "bench_bronze_native").strip()
DS_SILVER_BQ = os.getenv("DS_SILVER_BQ", "bench_silver_bq").strip()
DS_GOLD_BQ = os.getenv("DS_GOLD_BQ", "bench_gold_bq").strip()
DS_METRICS = os.getenv("DS_METRICS", "bench_metrics").strip()

SCALE_PRESET = os.getenv("SCALE_PRESET", "small_1g").strip()
if SCALE_PRESET not in {"small_1g", "medium_100g", "large_500g"}:
    raise ValueError(f"Unsupported SCALE_PRESET: {SCALE_PRESET!r}")

BF_PASS_FILTER_RAW = (os.getenv("BF_PASS_FILTER") or "all").strip().lower()
SELECTED_BF_PASSES = parse_bf_pass_filter(BF_PASS_FILTER_RAW)

RUN_PYTHON_UDF_STAGE = _parse_bool_env("RUN_PYTHON_UDF_STAGE", True)
RUN_BQ_MANAGED_UDF = _parse_bool_env("RUN_BQ_MANAGED_UDF", False)
WRITE_METRICS_TO_BQ = _parse_bool_env("WRITE_METRICS_TO_BQ", True)
METRICS_JSONL_PATH = REPO_ROOT / "results" / "live_runs" / "benchmark_runs.jsonl"

_default_run_id = (
    f"bf-{SCALE_PRESET}-"
    f"{datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')}-"
    f"{uuid.uuid4().hex[:6]}"
)
RUN_ID = (os.getenv("BENCH_RUN_ID") or os.getenv("RUN_ID") or "").strip() or _default_run_id

print(f"Repository root       : {REPO_ROOT}")
print(f"RUN_ID                : {RUN_ID}")
print(f"PROJECT_ID / LOCATION : {PROJECT_ID} / {BQ_LOCATION} ({BQ_INFO_SCHEMA_REGION})")
print(f"Reservation           : {BQ_RESERVATION_NAME} ({BQ_SLOT_CAP} slots | allow_unreserved={ALLOW_UNRESERVED_BQ})")
print(f"SCALE_PRESET          : {SCALE_PRESET}")
print(f"BF_PASS_FILTER        : {BF_PASS_FILTER_RAW} -> {[p for p in VALID_BF_PASS_KEYS if p in SELECTED_BF_PASSES]}")
print(f"Bronze BigLake / Nat. : {DS_BRONZE_BIGLAKE} / {DS_BRONZE_NATIVE}")
print(f"Silver / Gold / Metr. : {DS_SILVER_BQ} / {DS_GOLD_BQ} / {DS_METRICS}")
print(f"Stage 2B UDF / Managed: {RUN_PYTHON_UDF_STAGE} / {RUN_BQ_MANAGED_UDF}")

Repository root       : .
RUN_ID                : 20261116T103813Z_large_500g
PROJECT_ID / LOCATION : <your-gcp-project-id> / US (region-us)
Reservation           : bench-elt-slots (400 slots | allow_unreserved=False)
SCALE_PRESET          : large_500g
BF_PASS_FILTER        : all -> ['biglake', 'native', 'ctas_native', 'lazy_e2e']
Bronze BigLake / Nat. : bench_bronze_biglake / bench_bronze_native
Silver / Gold / Metr. : bench_silver_bq / bench_gold_bq / bench_metrics
Stage 2B UDF / Managed: True / False


In [2]:
# Configure BigQuery DataFrames BEFORE starting the session so partial ordering applies to all queries.
bpd.close_session()
bpd.options.bigquery.project = PROJECT_ID
bpd.options.bigquery.location = BQ_LOCATION
bpd.options.bigquery.ordering_mode = "partial"

session = bpd.get_global_session()
assert bpd.options.bigquery.ordering_mode == "partial", (
    f"Expected bpd.options.bigquery.ordering_mode == 'partial', got {bpd.options.bigquery.ordering_mode!r}"
)

bq_client = bigquery.Client(project=PROJECT_ID, location=BQ_LOCATION)
print(
    f"Initialized BigFrames session={session.session_id} | "
    f"ordering_mode={bpd.options.bigquery.ordering_mode!r} | "
    f"bigframes={bpd.__version__}"
)


Initialized BigFrames session=session893eef | ordering_mode='partial' | bigframes=2.53.0


### Live Slot Reservation Governance Verification (`SPEC.md` §1.4)

Before timing benchmark stages, confirm two properties directly against BigQuery in `US`:
1. Normal analytical queries execute inside `BQ_RESERVATION_NAME` (`reservation_id` populated in `INFORMATION_SCHEMA.JOBS_BY_PROJECT`; BigQuery Editions reports informational `total_bytes_billed` >= 10 MiB while billing strictly by slot-hours).
2. Attempting `SET @@reservation = 'none'; SELECT 1;` is rejected (`400`/`403`) for standard job-user principals lacking `bigquery.reservations.*` override permissions (or detects truthfully if the current caller is a Sandbox `roles/owner` admin with override privileges).


In [3]:
def verify_slot_reservation_governance(
    client: bigquery.Client,
    project_id: str,
    ds_bronze_native: str,
    reservation_name: str,
    info_schema_region: str,
    allow_unreserved_bq: bool = False,
) -> dict[str, Any]:
    """Probe Enterprise reservation assignment and test On-Demand bypass behavior."""
    probe_sql = (
        f"SELECT COUNT(*) AS row_cnt, MIN(d_date_sk) AS min_sk, MAX(d_date_sk) AS max_sk "
        f"FROM `{project_id}.{ds_bronze_native}.date_dim`"
    )
    probe_job = client.query(probe_sql, job_config=bigquery.QueryJobConfig(use_query_cache=False))
    list(probe_job.result())

    audit_sql = f"""
    SELECT
      job_id,
      COALESCE(reservation_id, 'none') AS reservation_id,
      COALESCE(total_bytes_billed, 0) AS total_bytes_billed,
      COALESCE(total_bytes_processed, 0) AS total_bytes_processed,
      COALESCE(total_slot_ms, 0) AS total_slot_ms
    FROM `{info_schema_region}`.INFORMATION_SCHEMA.JOBS_BY_PROJECT
    WHERE creation_time >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 10 MINUTE)
      AND job_id = @probe_job_id
    """
    audit_cfg = bigquery.QueryJobConfig(
        query_parameters=[bigquery.ScalarQueryParameter("probe_job_id", "STRING", probe_job.job_id)],
        use_query_cache=False,
    )
    audit_rows = list(client.query(audit_sql, job_config=audit_cfg).result())
    if audit_rows:
        r = audit_rows[0]
        res_id = str(r.reservation_id)
        bytes_billed = int(r.total_bytes_billed)
        slot_ms = int(r.total_slot_ms)
    else:
        raw_res = getattr(probe_job, "reservation_id", None) or (
            getattr(probe_job, "_properties", {}) or {}
        ).get("statistics", {}).get("reservation_id")
        res_id = str(raw_res or "none")
        bytes_billed = int(probe_job.total_bytes_billed or 0)
        slot_ms = int(probe_job.slot_millis or 0)

    in_expected_res = (reservation_name in res_id) and (res_id != "none")
    edition = str(
        ((getattr(probe_job, "_properties", {}) or {}).get("statistics", {}) or {}).get("edition", "")
    )

    print(
        f"[Governance Probe] job_id={probe_job.job_id} | reservation_id={res_id} | "
        f"edition={edition or 'UNKNOWN'} | info_bytes_billed={bytes_billed} | total_slot_ms={slot_ms}"
    )
    if not in_expected_res:
        msg = (
            f"Probe query did not execute inside expected reservation {reservation_name!r} "
            f"(observed reservation_id={res_id!r}, edition={edition or 'UNKNOWN'}, "
            f"total_bytes_billed={bytes_billed})."
        )
        if not allow_unreserved_bq:
            raise RuntimeError(
                f"{msg} Failing closed to prevent unfair unreserved On-Demand execution; "
                "fix reservation assignment or explicitly set ALLOW_UNRESERVED_BQ='true'."
            )
        print(f"[WARN] ALLOW_UNRESERVED_BQ=true: {msg}")
    else:
        print(
            f"[PASS] Probe query executed inside assigned slot reservation {res_id!r} "
            f"(edition={edition or 'ENTERPRISE'}; billed by slot-hours, $0 On-Demand byte charges)."
        )

    # Test On-Demand bypass behavior truthfully without faking errors.
    bypass_status: str
    bypass_detail: str
    try:
        bypass_job = client.query(
            "SET @@reservation = 'none'; SELECT 1 AS bypass_probe;",
            job_config=bigquery.QueryJobConfig(use_query_cache=False),
        )
        list(bypass_job.result())
        bypass_res = getattr(bypass_job, "reservation_id", None) or "none"
        bypass_status = "WARN_SANDBOX_OVERRIDE_PRIVILEGE"
        bypass_detail = (
            f"Caller succeeded with SET @@reservation='none' (job_id={bypass_job.job_id}, "
            f"reservation_id={bypass_res}). This occurs when executing as a Sandbox roles/owner "
            "admin holding bigquery.reservations.* override permissions; production workloads must "
            "use roles/bigquery.jobUser (without reservation admin) to make slot caps non-bypassable."
        )
        print(f"[WARN] {bypass_detail}")
    except (gapi_exceptions.BadRequest, gapi_exceptions.Forbidden) as exc:
        bypass_status = "PASS_BYPASS_REJECTED"
        err_msg = getattr(exc, "message", None) or str(exc)
        bypass_detail = f"On-Demand reservation bypass rejected ({type(exc).__name__}): {err_msg}"
        print(f"[PASS] {bypass_detail}")

    return {
        "probe_job_id": probe_job.job_id,
        "reservation_id": res_id,
        "total_bytes_billed": bytes_billed,
        "in_expected_reservation": in_expected_res,
        "zero_bytes_billed": (bytes_billed == 0),
        "bypass_test_status": bypass_status,
        "bypass_test_detail": bypass_detail,
    }

In [4]:
def read_gbq_table_lazy(
    bf_session: Any,
    table_fqn: str,
    columns: list[str] | None = None,
) -> bpd.DataFrame:
    """Read a BigQuery Native or BigLake External table lazily without metadata cache reuse.

    BigQuery does not support FOR SYSTEM_TIME AS OF snapshots on standard GCS external
    tables; bigframes 2.53.0's public ``read_gbq_table`` time-travel check only catches
    ``NotFound``, so if dry-run validation raises ``400 BadRequest`` for BigLake tables,
    retry lazily via ``_loader.read_gbq_table(..., enable_snapshot=False)``.
    """
    cols = columns or ()
    try:
        return bpd.read_gbq_table(table_fqn, columns=cols, use_cache=False)
    except gapi_exceptions.BadRequest as exc:
        msg = str(exc).lower()
        if any(tok in msg for tok in ("time travel", "system_time", "external", "snapshot")):
            return bf_session._loader.read_gbq_table(
                table_id=table_fqn,
                columns=cols,
                use_cache=False,
                enable_snapshot=False,
            )
        raise


def get_session_job_ids(bf_session: Any) -> list[str]:
    """Return ordered BigQuery job_ids currently recorded in BigFrames session history."""
    hist_df = bf_session.execution_history(all_cells=True).to_dataframe()
    if hist_df.empty or "job_id" not in hist_df.columns:
        return []
    return [str(jid) for jid in hist_df["job_id"].dropna().tolist() if str(jid).strip()]


def query_stage_bq_telemetry(
    client: bigquery.Client,
    info_schema_region: str,
    start_epoch_sec: float,
    end_epoch_sec: float,
    stage_job_ids: list[str],
    stage_label_key: str,
    stage_label_val: str,
    reservation_name: str = "",
) -> dict[str, Any]:
    """Extract deduplicated slot_ms, bytes_processed, reservation_id, and BI Engine telemetry."""
    dedup_job_ids = list(
        dict.fromkeys(str(jid).strip() for jid in stage_job_ids if str(jid).strip())
    )
    params: list[Any] = [
        bigquery.ScalarQueryParameter("start_epoch", "INT64", int(start_epoch_sec) - 10),
        bigquery.ScalarQueryParameter("end_epoch", "INT64", int(end_epoch_sec) + 30),
        bigquery.ScalarQueryParameter("lbl_key", "STRING", stage_label_key),
        bigquery.ScalarQueryParameter("lbl_val", "STRING", stage_label_val),
        bigquery.ScalarQueryParameter("reservation_name", "STRING", reservation_name),
    ]
    label_clause = (
        "(job_type = 'QUERY' AND COALESCE(statement_type, '') != 'SCRIPT' "
        "AND EXISTS (SELECT 1 FROM UNNEST(labels) AS lbl "
        "WHERE lbl.key = @lbl_key AND lbl.value = @lbl_val))"
    )
    if dedup_job_ids:
        filter_clause = (
            f"AND COALESCE(statement_type, '') != 'SCRIPT' "
            f"AND (job_id IN UNNEST(@stage_job_ids) OR {label_clause})"
        )
        params.append(bigquery.ArrayQueryParameter("stage_job_ids", "STRING", dedup_job_ids))
    else:
        filter_clause = f"AND {label_clause}"

    sql = f"""
    WITH stage_jobs AS (
      SELECT
        job_id,
        COALESCE(total_slot_ms, 0) AS total_slot_ms,
        COALESCE(total_bytes_processed, 0) AS total_bytes_processed,
        COALESCE(total_bytes_billed, 0) AS total_bytes_billed,
        NULLIF(TRIM(reservation_id), '') AS reservation_id,
        COALESCE(bi_engine_statistics.bi_engine_mode, 'NONE') AS bi_engine_mode,
        bi_engine_statistics.bi_engine_reasons AS bi_engine_reasons
      FROM `{info_schema_region}`.INFORMATION_SCHEMA.JOBS_BY_PROJECT
      WHERE creation_time BETWEEN TIMESTAMP_SECONDS(@start_epoch) AND TIMESTAMP_SECONDS(@end_epoch)
        AND user_email = SESSION_USER()
        {filter_clause}
      QUALIFY ROW_NUMBER() OVER (PARTITION BY job_id ORDER BY creation_time DESC) = 1
    ),
    agg_stats AS (
      SELECT
        COUNT(*) AS job_count,
        COUNTIF(reservation_id IS NULL) AS null_reservation_jobs,
        COUNTIF(
          reservation_id IS NULL
          OR (@reservation_name != '' AND STRPOS(reservation_id, @reservation_name) = 0)
        ) AS unreserved_jobs,
        COALESCE(SUM(total_slot_ms), 0) AS bq_total_slot_ms,
        COALESCE(SUM(total_bytes_processed), 0) AS bq_bytes_processed,
        COALESCE(SUM(total_bytes_billed), 0) AS bq_bytes_billed,
        ARRAY_AGG(DISTINCT reservation_id IGNORE NULLS) AS reservation_ids,
        ARRAY_AGG(DISTINCT bi_engine_mode IGNORE NULLS) AS bi_engine_modes
      FROM stage_jobs
    ),
    reason_codes AS (
      SELECT ARRAY_AGG(DISTINCT r.code IGNORE NULLS) AS bi_engine_reason_codes
      FROM stage_jobs AS sj, UNNEST(sj.bi_engine_reasons) AS r
    )
    SELECT
      agg_stats.job_count,
      agg_stats.null_reservation_jobs,
      agg_stats.unreserved_jobs,
      agg_stats.bq_total_slot_ms,
      agg_stats.bq_bytes_processed,
      agg_stats.bq_bytes_billed,
      agg_stats.reservation_ids,
      agg_stats.bi_engine_modes,
      reason_codes.bi_engine_reason_codes
    FROM agg_stats
    CROSS JOIN reason_codes
    """

    # Retry briefly outside the stage timer for INFORMATION_SCHEMA eventual visibility.
    row = None
    expected_min = len(dedup_job_ids) if dedup_job_ids else 1
    for attempt in range(3):
        rows = list(
            client.query(
                sql,
                job_config=bigquery.QueryJobConfig(query_parameters=params, use_query_cache=False),
            ).result()
        )
        row = rows[0] if rows else None
        if row is not None and int(row.job_count) >= expected_min:
            break
        if attempt < 2:
            time.sleep(2.0)

    if row is not None and int(row.job_count) > 0:
        res_list = list(row.reservation_ids or [])
        bi_modes = list(row.bi_engine_modes or [])
        bi_reasons = list(row.bi_engine_reason_codes or [])
        return {
            "job_count": int(row.job_count),
            "null_reservation_jobs": int(row.null_reservation_jobs),
            "unreserved_jobs": int(row.unreserved_jobs),
            "bq_total_slot_ms": int(row.bq_total_slot_ms),
            "bq_bytes_processed": int(row.bq_bytes_processed),
            "bq_bytes_billed": int(row.bq_bytes_billed),
            "reservation_id": ", ".join(res_list) if res_list else "none",
            "bi_engine_mode": ", ".join(bi_modes) if bi_modes else "NONE",
            "bi_engine_reasons": ", ".join(bi_reasons) if bi_reasons else "",
        }

    # Defensive fallback via Jobs API if INFORMATION_SCHEMA lags behind job completion.
    total_slot_ms = 0
    total_bytes_processed = 0
    total_bytes_billed = 0
    null_res_jobs = 0
    unreserved_jobs = 0
    res_set: set[str] = set()
    for jid in dedup_job_ids:
        job = client.get_job(jid)
        total_slot_ms += int(getattr(job, "slot_millis", 0) or 0)
        total_bytes_processed += int(getattr(job, "total_bytes_processed", 0) or 0)
        total_bytes_billed += int(getattr(job, "total_bytes_billed", 0) or 0)
        rid = getattr(job, "reservation_id", None) or (
            getattr(job, "_properties", {}) or {}
        ).get("statistics", {}).get("reservation_id")
        rid_str = str(rid).strip() if rid else ""
        if not rid_str:
            null_res_jobs += 1
            unreserved_jobs += 1
        else:
            res_set.add(rid_str)
            if reservation_name and reservation_name not in rid_str:
                unreserved_jobs += 1
    return {
        "job_count": len(dedup_job_ids),
        "null_reservation_jobs": null_res_jobs,
        "unreserved_jobs": unreserved_jobs,
        "bq_total_slot_ms": total_slot_ms,
        "bq_bytes_processed": total_bytes_processed,
        "bq_bytes_billed": total_bytes_billed,
        "reservation_id": ", ".join(sorted(res_set)) if res_set else "unknown",
        "bi_engine_mode": "UNKNOWN",
        "bi_engine_reasons": "",
    }


def fetch_materialized_checksum(
    client: bigquery.Client,
    table_fqn: str,
    checksum_col: str,
) -> tuple[int, float]:
    """Compute exact output row count and ROUND(SUM(checksum_col), 2) outside timers."""
    sql = f"""
    SELECT
      COUNT(*) AS output_row_count,
      ROUND(COALESCE(SUM(CAST({checksum_col} AS FLOAT64)), 0.0), 2) AS checksum_net_paid
    FROM `{table_fqn}`
    """
    rows = list(client.query(sql, job_config=bigquery.QueryJobConfig(use_query_cache=False)).result())
    row = rows[0]
    return int(row.output_row_count), float(row.checksum_net_paid or 0.0)


def compile_df_ctas_sql(
    df: bpd.DataFrame,
    dest_table: str,
    cluster_cols: list[str],
    expected_columns: list[str],
) -> str:
    """Compile a BigFrames DataFrame via df.sql into a single-pass clustered CTAS query."""
    clean_df = df.reset_index(drop=True) if df._has_index else df
    actual_cols = [str(col) for col in clean_df.columns]
    if actual_cols != list(expected_columns):
        raise ValueError(
            f"CTAS column mismatch for {dest_table}: got {actual_cols}, expected {expected_columns}"
        )
    inner_sql = clean_df.sql.strip().rstrip(";")
    select_cols_sql = ", ".join(f"`{col}`" for col in expected_columns)
    cluster_sql = ", ".join(cluster_cols)
    return (
        f"CREATE OR REPLACE TABLE `{dest_table}`\n"
        f"CLUSTER BY {cluster_sql}\n"
        f"AS\n"
        f"SELECT {select_cols_sql}\n"
        f"FROM (\n{inner_sql}\n) AS bf_compiled"
    )


def execute_df_sql_ctas(
    client: bigquery.Client,
    df: bpd.DataFrame,
    dest_table: str,
    cluster_cols: list[str],
    expected_columns: list[str],
    labels: dict[str, str],
    collected_job_ids: list[str] | None = None,
) -> str:
    """Execute a single-pass clustered CTAS directly via bigquery.Client using df.sql."""
    ctas_sql = compile_df_ctas_sql(
        df=df,
        dest_table=dest_table,
        cluster_cols=cluster_cols,
        expected_columns=expected_columns,
    )
    job_config = bigquery.QueryJobConfig(labels=labels, use_query_cache=False)
    ctas_job = client.query(ctas_sql, job_config=job_config)
    ctas_job.result()
    if collected_job_ids is not None and getattr(ctas_job, "job_id", None):
        collected_job_ids.append(str(ctas_job.job_id))
    return dest_table

### Medallion Pipeline Stage Definitions (`SPEC.md` §3 Parity Contract)

All pipeline functions below are implemented using genuine `bigframes.pandas` DataFrame/Series operations compiled lazily to BigQuery SQL in `ordering_mode="partial"`:
- **Stage 1 (`stage1_bronze_to_silver`):** Pre-aggregates `store_returns` and `web_returns` BEFORE left-joining `store_sales` and `web_sales` (anti-fan-out rule), filters clean sales lines (`sold_date_sk`, `item_sk`, `customer_sk` non-null, `quantity > 0`), unions via `bpd.concat(..., ignore_index=True)`, and materializes 15 Silver columns (clustered by `sold_date_sk, customer_sk, item_sk` in `ctas_native`).
- **Stage 2A (`stage2a_silver_vectorized`):** Computes `order_risk_tier` (`case_when` 4-branch predicate) and `adjusted_margin_score` rounded to 4 decimals via vectorized expressions.
- **Stage 2B (`stage2b_silver_udf`):** Deterministic sample filter (`100%` on `small_1g`, `(order_number.abs() % 20) == 0` 5% on `medium_100g`/`large_500g`) contrasting native Dremel SQL pushdown inside the slot reservation vs. optional `@bpd.udf` Managed Python UDF containers.
- **Stage 3 (`stage3_silver_to_gold`):** Joins `silver_enriched_sales_*` with the 4 dimension tables from the matching Bronze dataset, filters NULL join keys prior to `.merge(...)` for exact SQL `INNER JOIN` parity, optionally caches `daily.cache()` for `to_gbq` passes (omitted in `ctas_native` so BigQuery compiles the CTE + 7d/30d rolling windows in one Dremel query job), and materializes `gold_omnichannel_kpi_mart_*` (clustered by `d_date_sk, channel, i_category` in `ctas_native`).
- **Stage E2E (`stage_e2e_bronze_to_gold`):** Chains `_build_stage1_silver_df` -> `apply_vectorized_scoring` -> `_build_stage3_gold_df(..., use_daily_cache=False)` lazily over `DS_BRONZE_NATIVE` and materializes `gold_omnichannel_kpi_mart_lazy_e2e` in a single clustered `df.sql` CTAS job with zero intermediate Silver disk writes.

In [5]:
STAGE1_SILVER_COLUMNS = [
    "channel",
    "sold_date_sk",
    "item_sk",
    "customer_sk",
    "cdemo_sk",
    "order_number",
    "quantity",
    "gross_sales_price",
    "discount_amt",
    "net_paid",
    "net_profit",
    "return_qty",
    "return_amt",
    "net_paid_after_return",
    "net_profit_after_return",
]

STAGE_CLUSTER_KEYS: dict[str, list[str]] = {
    "stage1_bronze_to_silver": ["sold_date_sk", "customer_sk", "item_sk"],
    "stage2a_silver_vectorized": ["sold_date_sk", "customer_sk", "item_sk"],
    "stage2b_silver_udf": ["sold_date_sk", "customer_sk", "item_sk"],
    "stage3_silver_to_gold": ["d_date_sk", "channel", "i_category"],
    "stage_e2e_bronze_to_gold": ["d_date_sk", "channel", "i_category"],
}


def _build_channel_silver(
    bf_session: Any,
    project_id: str,
    bronze_ds: str,
    *,
    channel_label: str,
    sales_table: str,
    returns_table: str,
    s_prefix: str,
    r_prefix: str,
    customer_col: str,
    cdemo_col: str,
    order_col: str,
    ret_order_col: str,
) -> bpd.DataFrame:
    """Normalize one TPC-DS sales+returns channel into the 15-column Stage 1 schema."""
    sold_col = f"{s_prefix}sold_date_sk"
    item_col = f"{s_prefix}item_sk"
    qty_col = f"{s_prefix}quantity"
    ext_price_col = f"{s_prefix}ext_sales_price"
    ext_disc_col = f"{s_prefix}ext_discount_amt"
    net_paid_col = f"{s_prefix}net_paid"
    net_profit_col = f"{s_prefix}net_profit"

    r_item_col = f"{r_prefix}item_sk"
    r_qty_col = f"{r_prefix}return_quantity"
    r_amt_col = f"{r_prefix}return_amt"
    r_loss_col = f"{r_prefix}net_loss"

    sales_df = read_gbq_table_lazy(
        bf_session,
        f"{project_id}.{bronze_ds}.{sales_table}",
        columns=[
            sold_col,
            item_col,
            customer_col,
            cdemo_col,
            order_col,
            qty_col,
            ext_price_col,
            ext_disc_col,
            net_paid_col,
            net_profit_col,
        ],
    )
    sales_clean = sales_df[
        sales_df[sold_col].notna()
        & sales_df[item_col].notna()
        & sales_df[customer_col].notna()
        & (sales_df[qty_col] > 0)
    ]

    returns_df = read_gbq_table_lazy(
        bf_session,
        f"{project_id}.{bronze_ds}.{returns_table}",
        columns=[r_item_col, ret_order_col, r_qty_col, r_amt_col, r_loss_col],
    )
    returns_clean = returns_df[
        returns_df[r_item_col].notna() & returns_df[ret_order_col].notna()
    ].assign(
        **{
            r_qty_col: returns_df[r_qty_col].fillna(0),
            r_amt_col: returns_df[r_amt_col].fillna(0.0),
            r_loss_col: returns_df[r_loss_col].fillna(0.0),
        }
    )
    returns_agg = returns_clean.groupby(
        [r_item_col, ret_order_col], as_index=False, dropna=False
    ).agg(
        _agg_return_qty=(r_qty_col, "sum"),
        _agg_return_amt=(r_amt_col, "sum"),
        _agg_return_net_loss=(r_loss_col, "sum"),
    )

    joined = sales_clean.merge(
        returns_agg,
        how="left",
        left_on=[item_col, order_col],
        right_on=[r_item_col, ret_order_col],
    )

    normalized = joined.assign(
        channel=channel_label,
        sold_date_sk=joined[sold_col].astype("Int64"),
        item_sk=joined[item_col].astype("Int64"),
        customer_sk=joined[customer_col].astype("Int64"),
        cdemo_sk=joined[cdemo_col].astype("Int64"),
        order_number=joined[order_col].astype("Int64"),
        quantity=joined[qty_col].astype("Int64"),
        gross_sales_price=joined[ext_price_col].fillna(0.0).astype("Float64"),
        discount_amt=joined[ext_disc_col].fillna(0.0).astype("Float64"),
        net_paid=joined[net_paid_col].fillna(0.0).astype("Float64"),
        net_profit=joined[net_profit_col].fillna(0.0).astype("Float64"),
        return_qty=joined["_agg_return_qty"].fillna(0).astype("Int64"),
        return_amt=joined["_agg_return_amt"].fillna(0.0).astype("Float64"),
        _return_net_loss=joined["_agg_return_net_loss"].fillna(0.0).astype("Float64"),
    )
    normalized = normalized.assign(
        net_paid_after_return=(normalized["net_paid"] - normalized["return_amt"]).astype("Float64"),
        net_profit_after_return=(
            normalized["net_profit"] - normalized["_return_net_loss"]
        ).astype("Float64"),
    )
    return normalized[STAGE1_SILVER_COLUMNS]


def _build_stage1_silver_df(
    bf_session: Any,
    project_id: str,
    bronze_ds: str,
) -> bpd.DataFrame:
    """Lazily build deduplicated store + web omnichannel Silver DataFrame."""
    store_df = _build_channel_silver(
        bf_session,
        project_id,
        bronze_ds,
        channel_label="STORE",
        sales_table="store_sales",
        returns_table="store_returns",
        s_prefix="ss_",
        r_prefix="sr_",
        customer_col="ss_customer_sk",
        cdemo_col="ss_cdemo_sk",
        order_col="ss_ticket_number",
        ret_order_col="sr_ticket_number",
    )
    web_df = _build_channel_silver(
        bf_session,
        project_id,
        bronze_ds,
        channel_label="WEB",
        sales_table="web_sales",
        returns_table="web_returns",
        s_prefix="ws_",
        r_prefix="wr_",
        customer_col="ws_bill_customer_sk",
        cdemo_col="ws_bill_cdemo_sk",
        order_col="ws_order_number",
        ret_order_col="wr_order_number",
    )
    return bpd.concat([store_df, web_df], ignore_index=True)[STAGE1_SILVER_COLUMNS]


def stage1_bronze_to_silver(
    bf_session: Any,
    project_id: str,
    bronze_ds: str,
    silver_ds: str,
    suffix: str,
    labels: dict[str, str],
    *,
    client: bigquery.Client | None = None,
    use_ctas: bool = False,
    collected_job_ids: list[str] | None = None,
) -> tuple[str, float]:
    """Stage 1: Deduplicated store + web sales with pre-aggregated returns -> Silver."""
    silver_df = _build_stage1_silver_df(bf_session, project_id, bronze_ds)
    dest_table = f"{project_id}.{silver_ds}.silver_omnichannel_sales_{suffix}"
    if use_ctas:
        if client is None:
            raise ValueError("bigquery.Client is required when use_ctas=True")
        execute_df_sql_ctas(
            client=client,
            df=silver_df,
            dest_table=dest_table,
            cluster_cols=STAGE_CLUSTER_KEYS["stage1_bronze_to_silver"],
            expected_columns=STAGE1_SILVER_COLUMNS,
            labels=labels,
            collected_job_ids=collected_job_ids,
        )
    else:
        silver_df.to_gbq(dest_table, if_exists="replace", index=False, labels=labels)
    return dest_table, 1.0

In [6]:
STAGE2_ENRICHED_COLUMNS = STAGE1_SILVER_COLUMNS + [
    "order_risk_tier",
    "adjusted_margin_score",
]


def apply_vectorized_scoring(df: bpd.DataFrame) -> bpd.DataFrame:
    """Apply SPEC.md Stage 2A/2B scoring rules cleanly as compiled Dremel GoogleSQL pushdown."""
    high_risk = (df["return_amt"] > 0.5 * df["gross_sales_price"]) & (df["return_qty"] > 0)
    vip_full = (df["net_profit_after_return"] > 50.0) & (df["discount_amt"] == 0.0)
    promo = df["discount_amt"] > 0.25 * df["gross_sales_price"]

    order_risk_tier = df["channel"].case_when(
        [
            (high_risk, "HIGH_RETURN_RISK"),
            (vip_full, "VIP_FULL_PRICE"),
            (promo, "PROMO_DRIVEN"),
            (True, "STANDARD"),
        ]
    )
    adjusted_margin_score = (
        df["net_profit_after_return"]
        - (0.15 * df["return_amt"])
        + (0.05 * df["net_paid_after_return"])
    ).round(4)
    return df.assign(
        order_risk_tier=order_risk_tier,
        adjusted_margin_score=adjusted_margin_score,
    )[STAGE2_ENRICHED_COLUMNS]


def stage2a_silver_vectorized(
    bf_session: Any,
    project_id: str,
    silver_ds: str,
    suffix: str,
    labels: dict[str, str],
    *,
    client: bigquery.Client | None = None,
    use_ctas: bool = False,
    collected_job_ids: list[str] | None = None,
) -> tuple[str, float]:
    """Stage 2A: Pure vectorized SQL pushdown enrichment on materialized Silver."""
    silver_src = read_gbq_table_lazy(
        bf_session,
        f"{project_id}.{silver_ds}.silver_omnichannel_sales_{suffix}",
    )
    enriched = apply_vectorized_scoring(silver_src)
    dest_table = f"{project_id}.{silver_ds}.silver_enriched_sales_{suffix}"
    if use_ctas:
        if client is None:
            raise ValueError("bigquery.Client is required when use_ctas=True")
        execute_df_sql_ctas(
            client=client,
            df=enriched,
            dest_table=dest_table,
            cluster_cols=STAGE_CLUSTER_KEYS["stage2a_silver_vectorized"],
            expected_columns=STAGE2_ENRICHED_COLUMNS,
            labels=labels,
            collected_job_ids=collected_job_ids,
        )
    else:
        enriched.to_gbq(dest_table, if_exists="replace", index=False, labels=labels)
    return dest_table, 1.0

In [7]:
def _apply_bq_managed_python_udf_scoring(
    df: bpd.DataFrame,
    silver_ds: str,
) -> bpd.DataFrame:
    """Optional Stage 2B toggle: execute BigQuery Managed Python UDFs via @bpd.udf.

    Unlike native Dremel SQL pushdown (which executes strictly inside BQ_RESERVATION_NAME
    at $0 extra SKU) and @bpd.remote_function (which calls an external Cloud Run HTTP
    service), @bpd.udf deploys container-backed BigQuery Managed Python routines billed
    separately under the BigQuery Services SKU outside slot reservations.
    """

    @bpd.udf(
        input_types=[float, float, int, float, float],
        output_type=str,
        dataset=silver_ds,
    )
    def udf_order_risk_tier(
        return_amt: float,
        gross_sales_price: float,
        return_qty: int,
        net_profit_after_return: float,
        discount_amt: float,
    ) -> str:
        r_amt = float(return_amt or 0.0)
        gross = float(gross_sales_price or 0.0)
        r_qty = int(return_qty or 0)
        net_prof = float(net_profit_after_return or 0.0)
        disc = float(discount_amt or 0.0)
        if r_amt > 0.5 * gross and r_qty > 0:
            return "HIGH_RETURN_RISK"
        elif net_prof > 50.0 and disc == 0.0:
            return "VIP_FULL_PRICE"
        elif disc > 0.25 * gross:
            return "PROMO_DRIVEN"
        return "STANDARD"

    @bpd.udf(
        input_types=[float, float, float],
        output_type=float,
        dataset=silver_ds,
    )
    def udf_adjusted_margin_score(
        net_profit_after_return: float,
        return_amt: float,
        net_paid_after_return: float,
    ) -> float:
        net_prof = float(net_profit_after_return or 0.0)
        r_amt = float(return_amt or 0.0)
        net_paid = float(net_paid_after_return or 0.0)
        return round(net_prof - (0.15 * r_amt) + (0.05 * net_paid), 4)

    tier_inputs = df[
        ["return_amt", "gross_sales_price", "return_qty", "net_profit_after_return", "discount_amt"]
    ].assign(
        return_amt=df["return_amt"].astype("Float64"),
        gross_sales_price=df["gross_sales_price"].astype("Float64"),
        return_qty=df["return_qty"].astype("Int64"),
        net_profit_after_return=df["net_profit_after_return"].astype("Float64"),
        discount_amt=df["discount_amt"].astype("Float64"),
    )
    margin_inputs = df[
        ["net_profit_after_return", "return_amt", "net_paid_after_return"]
    ].assign(
        net_profit_after_return=df["net_profit_after_return"].astype("Float64"),
        return_amt=df["return_amt"].astype("Float64"),
        net_paid_after_return=df["net_paid_after_return"].astype("Float64"),
    )
    tier_series = tier_inputs.apply(udf_order_risk_tier, axis=1)
    margin_series = margin_inputs.apply(udf_adjusted_margin_score, axis=1)
    return df.assign(order_risk_tier=tier_series, adjusted_margin_score=margin_series)[
        STAGE2_ENRICHED_COLUMNS
    ]


def stage2b_silver_udf(
    bf_session: Any,
    project_id: str,
    silver_ds: str,
    suffix: str,
    scale_preset: str,
    run_bq_managed_udf: bool,
    labels: dict[str, str],
    *,
    client: bigquery.Client | None = None,
    use_ctas: bool = False,
    collected_job_ids: list[str] | None = None,
) -> tuple[str, float]:
    """Stage 2B: Deterministic row sampling + Python business rules (SQL pushdown vs @bpd.udf)."""
    silver_src = read_gbq_table_lazy(
        bf_session,
        f"{project_id}.{silver_ds}.silver_omnichannel_sales_{suffix}",
    )
    if scale_preset == "small_1g":
        sampled = silver_src
        sample_fraction = 1.0
    else:
        sampled = silver_src[(silver_src["order_number"].abs() % 20) == 0]
        sample_fraction = 0.05

    if run_bq_managed_udf:
        out_df = _apply_bq_managed_python_udf_scoring(sampled, silver_ds=silver_ds)
    else:
        out_df = apply_vectorized_scoring(sampled)

    dest_table = f"{project_id}.{silver_ds}.silver_udf_benchmark_out_{suffix}"
    if use_ctas:
        if client is None:
            raise ValueError("bigquery.Client is required when use_ctas=True")
        execute_df_sql_ctas(
            client=client,
            df=out_df,
            dest_table=dest_table,
            cluster_cols=STAGE_CLUSTER_KEYS["stage2b_silver_udf"],
            expected_columns=STAGE2_ENRICHED_COLUMNS,
            labels=labels,
            collected_job_ids=collected_job_ids,
        )
    else:
        out_df.to_gbq(dest_table, if_exists="replace", index=False, labels=labels)
    return dest_table, sample_fraction

In [8]:
STAGE3_INPUT_COLUMNS = [
    "channel",
    "sold_date_sk",
    "item_sk",
    "customer_sk",
    "cdemo_sk",
    "gross_sales_price",
    "return_amt",
    "net_paid_after_return",
    "net_profit_after_return",
    "order_risk_tier",
    "adjusted_margin_score",
]

STAGE3_GOLD_COLUMNS = [
    "d_date_sk",
    "d_date",
    "d_year",
    "d_moy",
    "channel",
    "i_category",
    "cd_credit_rating",
    "order_risk_tier",
    "order_lines",
    "unique_customers",
    "daily_gross_sales",
    "daily_return_amt",
    "daily_net_paid",
    "daily_net_profit",
    "avg_margin_score",
    "rolling_7d_net_paid",
    "rolling_30d_net_paid",
    "rolling_30d_return_rate",
    "category_daily_share",
]


def _build_stage3_gold_df(
    bf_session: Any,
    project_id: str,
    bronze_ds: str,
    enriched_df: bpd.DataFrame,
    *,
    use_daily_cache: bool,
) -> bpd.DataFrame:
    """Build the 5-way Star Schema join + 7d/30d rolling KPI DataFrame lazily."""
    enriched = enriched_df[STAGE3_INPUT_COLUMNS]
    customer = read_gbq_table_lazy(
        bf_session,
        f"{project_id}.{bronze_ds}.customer",
        columns=["c_customer_sk", "c_current_cdemo_sk"],
    )
    cust_demo = read_gbq_table_lazy(
        bf_session,
        f"{project_id}.{bronze_ds}.customer_demographics",
        columns=["cd_demo_sk", "cd_credit_rating"],
    )
    item = read_gbq_table_lazy(
        bf_session,
        f"{project_id}.{bronze_ds}.item",
        columns=["i_item_sk", "i_category"],
    )
    date_dim = read_gbq_table_lazy(
        bf_session,
        f"{project_id}.{bronze_ds}.date_dim",
        columns=["d_date_sk", "d_date", "d_year", "d_moy"],
    )

    # Filter non-null join keys and filter predicates before merging so BigFrames achieves
    # 100% SQL INNER JOIN semantics (bpd.DataFrame.merge otherwise treats NULL == NULL as a match).
    enriched = enriched[
        enriched["customer_sk"].notna()
        & enriched["item_sk"].notna()
        & enriched["sold_date_sk"].notna()
    ]
    customer = customer[customer["c_customer_sk"].notna()]
    cust_demo = cust_demo[
        cust_demo["cd_demo_sk"].notna() & cust_demo["cd_credit_rating"].notna()
    ]
    item = item[item["i_item_sk"].notna() & item["i_category"].notna()]
    date_dim = date_dim[date_dim["d_date_sk"].notna() & date_dim["d_date"].notna()]

    joined = enriched.merge(
        customer, how="inner", left_on="customer_sk", right_on="c_customer_sk"
    )
    joined = joined.assign(
        effective_cdemo_sk=joined["cdemo_sk"].fillna(joined["c_current_cdemo_sk"])
    )
    joined = joined[joined["effective_cdemo_sk"].notna()]
    joined = joined.merge(
        cust_demo, how="inner", left_on="effective_cdemo_sk", right_on="cd_demo_sk"
    )
    joined = joined.merge(item, how="inner", left_on="item_sk", right_on="i_item_sk")
    joined = joined.merge(date_dim, how="inner", left_on="sold_date_sk", right_on="d_date_sk")

    grain_cols = [
        "d_date_sk",
        "d_date",
        "d_year",
        "d_moy",
        "channel",
        "i_category",
        "cd_credit_rating",
        "order_risk_tier",
    ]
    daily = joined.groupby(grain_cols, as_index=False, dropna=False).agg(
        order_lines=("channel", "count"),
        unique_customers=("customer_sk", "nunique"),
        daily_gross_sales=("gross_sales_price", "sum"),
        daily_return_amt=("return_amt", "sum"),
        daily_net_paid=("net_paid_after_return", "sum"),
        daily_net_profit=("net_profit_after_return", "sum"),
        avg_margin_score=("adjusted_margin_score", "mean"),
    )

    if use_daily_cache:
        # Used by Passes 4 & 5 to materialize the daily aggregate into a temp table.
        # Omitted in Passes 6 & 7 so CTE + rolling windows compile into a single Dremel query job.
        daily = daily.cache()

    part_cols = ["channel", "i_category", "cd_credit_rating", "order_risk_tier"]
    d_idx = daily.sort_values("d_date_sk", ascending=True).set_index(
        ["d_date_sk", *part_cols], drop=True
    )
    r7 = (
        d_idx.groupby(part_cols, dropna=False)["daily_net_paid"]
        .rolling(7, min_periods=1)
        .sum()
        .droplevel([0, 1, 2, 3])
    )
    r30 = (
        d_idx.groupby(part_cols, dropna=False)[
            ["daily_net_paid", "daily_return_amt", "daily_gross_sales"]
        ]
        .rolling(30, min_periods=1)
        .sum()
        .droplevel([0, 1, 2, 3])
    )
    d_win = d_idx.assign(
        rolling_7d_net_paid=r7,
        rolling_30d_net_paid=r30["daily_net_paid"],
        _r30_return_amt=r30["daily_return_amt"],
        _r30_gross_sales=r30["daily_gross_sales"],
    ).reset_index(drop=False)

    chan_daily = daily.groupby(["d_date_sk", "channel"], as_index=False, dropna=False).agg(
        _channel_daily_net_paid=("daily_net_paid", "sum"),
    )
    gold_df = d_win.merge(chan_daily, how="inner", on=["d_date_sk", "channel"])
    gold_df = gold_df.assign(
        rolling_30d_return_rate=(
            gold_df["_r30_return_amt"]
            / gold_df["_r30_gross_sales"].where(gold_df["_r30_gross_sales"] != 0)
        ).astype("Float64"),
        category_daily_share=(
            gold_df["daily_net_paid"]
            / gold_df["_channel_daily_net_paid"].where(gold_df["_channel_daily_net_paid"] != 0)
        ).astype("Float64"),
    )[STAGE3_GOLD_COLUMNS]
    return gold_df


def stage3_silver_to_gold(
    bf_session: Any,
    project_id: str,
    bronze_ds: str,
    silver_ds: str,
    gold_ds: str,
    suffix: str,
    labels: dict[str, str],
    *,
    client: bigquery.Client | None = None,
    use_ctas: bool = False,
    use_daily_cache: bool = True,
    collected_job_ids: list[str] | None = None,
) -> tuple[str, float]:
    """Stage 3: 5-way Star Schema join against matching Bronze storage + rolling KPI windows."""
    enriched = read_gbq_table_lazy(
        bf_session,
        f"{project_id}.{silver_ds}.silver_enriched_sales_{suffix}",
        columns=STAGE3_INPUT_COLUMNS,
    )
    gold_df = _build_stage3_gold_df(
        bf_session,
        project_id,
        bronze_ds,
        enriched,
        use_daily_cache=use_daily_cache,
    )
    dest_table = f"{project_id}.{gold_ds}.gold_omnichannel_kpi_mart_{suffix}"
    if use_ctas:
        if client is None:
            raise ValueError("bigquery.Client is required when use_ctas=True")
        execute_df_sql_ctas(
            client=client,
            df=gold_df,
            dest_table=dest_table,
            cluster_cols=STAGE_CLUSTER_KEYS["stage3_silver_to_gold"],
            expected_columns=STAGE3_GOLD_COLUMNS,
            labels=labels,
            collected_job_ids=collected_job_ids,
        )
    else:
        gold_df.to_gbq(dest_table, if_exists="replace", index=False, labels=labels)
    return dest_table, 1.0


def stage_e2e_bronze_to_gold(
    bf_session: Any,
    project_id: str,
    bronze_ds: str,
    gold_ds: str,
    suffix: str,
    labels: dict[str, str],
    *,
    client: bigquery.Client,
    collected_job_ids: list[str] | None = None,
) -> tuple[str, float]:
    """Pass 7: Single-pass lazy Bronze -> Silver -> Vectorized -> Gold clustered df.sql CTAS."""
    silver_df = _build_stage1_silver_df(bf_session, project_id, bronze_ds)
    enriched_df = apply_vectorized_scoring(silver_df)
    gold_df = _build_stage3_gold_df(
        bf_session,
        project_id,
        bronze_ds,
        enriched_df,
        use_daily_cache=False,
    )
    dest_table = f"{project_id}.{gold_ds}.gold_omnichannel_kpi_mart_{suffix}"
    execute_df_sql_ctas(
        client=client,
        df=gold_df,
        dest_table=dest_table,
        cluster_cols=STAGE_CLUSTER_KEYS["stage_e2e_bronze_to_gold"],
        expected_columns=STAGE3_GOLD_COLUMNS,
        labels=labels,
        collected_job_ids=collected_job_ids,
    )
    return dest_table, 1.0

In [9]:
def _execute_and_log_stage(
    *,
    bf_session: Any,
    client: bigquery.Client,
    engine_variant: str,
    storage_backend: str,
    stage_id: str,
    checksum_col: str,
    stage_fn: Any,
    stage_note: str,
    silver_ds: str | None = None,
    suffix: str = "",
    silver_row_count: int | None = None,
) -> tuple[dict[str, Any], int | None]:
    """Run one timed stage, verify strict slot reservation governance, and log telemetry."""
    stage_tag = _sanitize_bq_label(f"{RUN_ID}-{engine_variant}-{stage_id}")
    stage_labels = {
        "bench_run_id": _sanitize_bq_label(RUN_ID),
        "engine_variant": _sanitize_bq_label(engine_variant),
        "stage_id": _sanitize_bq_label(stage_id),
        "bench_stage_key": stage_tag,
    }
    jobs_before = set(get_session_job_ids(bf_session))
    direct_job_ids: list[str] = []
    bpd.options.compute.extra_query_labels.clear()
    bpd.options.compute.assign_extra_query_labels(**stage_labels)

    start_epoch_sec = time.time()
    try:
        dest_table, nominal_sample_fraction = stage_fn(stage_labels, direct_job_ids)
    finally:
        bpd.options.compute.extra_query_labels.clear()
    end_epoch_sec = time.time()

    jobs_after = get_session_job_ids(bf_session)
    session_delta_ids = [jid for jid in jobs_after if jid not in jobs_before]
    stage_job_ids = list(dict.fromkeys(direct_job_ids + session_delta_ids))

    # Post-stage telemetry and verification execute STRICTLY outside the timed block.
    bq_meta = query_stage_bq_telemetry(
        client=client,
        info_schema_region=BQ_INFO_SCHEMA_REGION,
        start_epoch_sec=start_epoch_sec,
        end_epoch_sec=end_epoch_sec,
        stage_job_ids=stage_job_ids,
        stage_label_key="bench_stage_key",
        stage_label_val=stage_tag,
        reservation_name=BQ_RESERVATION_NAME,
    )
    assert bq_meta["job_count"] > 0, (
        f"Telemetry lookup returned 0 BigQuery jobs for {engine_variant} / {stage_id}"
    )

    in_stage_res = (
        (bq_meta["null_reservation_jobs"] == 0)
        and (bq_meta["unreserved_jobs"] == 0)
        and (BQ_RESERVATION_NAME in bq_meta["reservation_id"])
        and (bq_meta["reservation_id"] not in {"none", "unknown"})
    )
    if not in_stage_res:
        if not ALLOW_UNRESERVED_BQ:
            raise RuntimeError(
                f"Stage {engine_variant} / {stage_id} executed outside expected reservation "
                f"{BQ_RESERVATION_NAME!r}: observed reservation_id={bq_meta['reservation_id']!r}, "
                f"null_reservation_jobs={bq_meta['null_reservation_jobs']}, "
                f"unreserved_jobs={bq_meta['unreserved_jobs']}."
            )
        effective_bq_slot_cap = None
        unreserved_tag = "unreserved_on_demand"
        base_note = f"{stage_note}; {unreserved_tag}" if stage_note else unreserved_tag
    else:
        effective_bq_slot_cap = BQ_SLOT_CAP
        base_note = stage_note

    output_row_count, checksum_net_paid = fetch_materialized_checksum(
        client=client,
        table_fqn=dest_table,
        checksum_col=checksum_col,
    )
    assert output_row_count > 0, f"{stage_id} materialized 0 rows into {dest_table}"

    if stage_id == "stage1_bronze_to_silver":
        silver_row_count = output_row_count
    if stage_id == "stage2b_silver_udf":
        if silver_row_count is None:
            assert silver_ds is not None
            silver_src_fqn = f"{PROJECT_ID}.{silver_ds}.silver_omnichannel_sales_{suffix}"
            silver_row_count, _ = fetch_materialized_checksum(
                client=client,
                table_fqn=silver_src_fqn,
                checksum_col="net_paid_after_return",
            )
        assert silver_row_count > 0, "Expected non-zero Silver row count before Stage 2B"
        sample_fraction = round(output_row_count / float(silver_row_count), 6)
    else:
        sample_fraction = nominal_sample_fraction

    note = base_note
    if bq_meta["bi_engine_mode"] not in {"NONE", "UNKNOWN"}:
        bi_tag = f"bi_engine={bq_meta['bi_engine_mode']}"
        note = f"{note}; {bi_tag}" if note else bi_tag

    record = log_stage_metric(
        run_id=RUN_ID,
        scale_preset=SCALE_PRESET,
        engine_variant=engine_variant,
        storage_backend=storage_backend,
        stage_id=stage_id,
        start_epoch_sec=start_epoch_sec,
        end_epoch_sec=end_epoch_sec,
        output_row_count=output_row_count,
        checksum_net_paid=checksum_net_paid,
        worker_machine_type=None,
        total_worker_vcpus=None,
        bq_slot_cap=effective_bq_slot_cap,
        bq_total_slot_ms=bq_meta["bq_total_slot_ms"],
        bq_bytes_processed=bq_meta["bq_bytes_processed"],
        nqe_enabled=False,
        nqe_fallback_note=note,
        est_list_cost_usd=None,
        sample_fraction=sample_fraction,
        jsonl_path=METRICS_JSONL_PATH,
        write_to_bq=WRITE_METRICS_TO_BQ,
        project_id=PROJECT_ID,
        ds_metrics=DS_METRICS,
    )
    combined = {**record, **bq_meta, "dest_table": dest_table}
    print(
        f"[{engine_variant} | {stage_id}] "
        f"wall_clock={record['wall_clock_sec']:.2f}s | rows={output_row_count:,} | "
        f"sample_frac={record['sample_fraction']:.6f} | "
        f"checksum={checksum_net_paid:,.2f} | slot_ms={bq_meta['bq_total_slot_ms']:,} | "
        f"bytes={bq_meta['bq_bytes_processed']:,} | res={bq_meta['reservation_id']} | "
        f"bi={bq_meta['bi_engine_mode']}"
    )
    return combined, silver_row_count


def run_medallion_pass(
    *,
    bf_session: Any,
    client: bigquery.Client,
    engine_variant: str,
    storage_backend: str,
    bronze_ds: str,
    silver_ds: str,
    gold_ds: str,
    suffix: str,
    use_ctas: bool = False,
    use_daily_cache: bool = True,
) -> list[dict[str, Any]]:
    """Execute Stages 1 -> 2A -> (optional 2B) -> 3 for one Bronze storage/write pass."""
    mode_desc = "df.sql clustered CTAS" if use_ctas else "to_gbq(replace)"
    print(
        f"\n=== Starting {engine_variant} ({storage_backend} | {mode_desc}) | "
        f"bronze={bronze_ds} -> silver={silver_ds}, gold={gold_ds} (suffix=_{suffix}) ==="
    )

    stage_specs: list[tuple[str, str, Any, str]] = [
        (
            "stage1_bronze_to_silver",
            "net_paid_after_return",
            lambda lbl, jids: stage1_bronze_to_silver(
                bf_session,
                PROJECT_ID,
                bronze_ds,
                silver_ds,
                suffix,
                lbl,
                client=client,
                use_ctas=use_ctas,
                collected_job_ids=jids,
            ),
            "",
        ),
        (
            "stage2a_silver_vectorized",
            "net_paid_after_return",
            lambda lbl, jids: stage2a_silver_vectorized(
                bf_session,
                PROJECT_ID,
                silver_ds,
                suffix,
                lbl,
                client=client,
                use_ctas=use_ctas,
                collected_job_ids=jids,
            ),
            "",
        ),
    ]
    if RUN_PYTHON_UDF_STAGE:
        stage2b_note = (
            "bq_managed_python_udf_outside_reservation"
            if RUN_BQ_MANAGED_UDF
            else "sql_pushdown_in_reservation"
        )
        stage_specs.append(
            (
                "stage2b_silver_udf",
                "net_paid_after_return",
                lambda lbl, jids: stage2b_silver_udf(
                    bf_session,
                    PROJECT_ID,
                    silver_ds,
                    suffix,
                    SCALE_PRESET,
                    RUN_BQ_MANAGED_UDF,
                    lbl,
                    client=client,
                    use_ctas=use_ctas,
                    collected_job_ids=jids,
                ),
                stage2b_note,
            )
        )
    stage_specs.append(
        (
            "stage3_silver_to_gold",
            "daily_net_paid",
            lambda lbl, jids: stage3_silver_to_gold(
                bf_session,
                PROJECT_ID,
                bronze_ds,
                silver_ds,
                gold_ds,
                suffix,
                lbl,
                client=client,
                use_ctas=use_ctas,
                use_daily_cache=use_daily_cache,
                collected_job_ids=jids,
            ),
            "",
        )
    )

    results: list[dict[str, Any]] = []
    silver_row_count: int | None = None
    for stage_id, checksum_col, stage_fn, stage_note in stage_specs:
        combined, silver_row_count = _execute_and_log_stage(
            bf_session=bf_session,
            client=client,
            engine_variant=engine_variant,
            storage_backend=storage_backend,
            stage_id=stage_id,
            checksum_col=checksum_col,
            stage_fn=stage_fn,
            stage_note=stage_note,
            silver_ds=silver_ds,
            suffix=suffix,
            silver_row_count=silver_row_count,
        )
        results.append(combined)
    return results


def run_lazy_e2e_pass(
    *,
    bf_session: Any,
    client: bigquery.Client,
    engine_variant: str,
    storage_backend: str,
    bronze_ds: str,
    gold_ds: str,
    suffix: str,
) -> list[dict[str, Any]]:
    """Execute Pass 7: Single-pass lazy Bronze -> Gold df.sql CTAS with zero Silver writes."""
    print(
        f"\n=== Starting {engine_variant} ({storage_backend} | lazy E2E df.sql CTAS) | "
        f"bronze={bronze_ds} -> gold={gold_ds} (suffix=_{suffix}) ==="
    )
    combined, _ = _execute_and_log_stage(
        bf_session=bf_session,
        client=client,
        engine_variant=engine_variant,
        storage_backend=storage_backend,
        stage_id="stage_e2e_bronze_to_gold",
        checksum_col="daily_net_paid",
        stage_fn=lambda lbl, jids: stage_e2e_bronze_to_gold(
            bf_session,
            PROJECT_ID,
            bronze_ds,
            gold_ds,
            suffix,
            lbl,
            client=client,
            collected_job_ids=jids,
        ),
        stage_note="",
    )
    return [combined]

### Execute Governance Verification + Selected BigFrames Passes (Passes 4–7)

Runs inside `try ... finally: bpd.close_session()` so all temporary session-scoped `_anonymous_*` tables/routines are cleaned up immediately upon completion or error. Use `BF_PASS_FILTER` (default `"all"`, or e.g. `"ctas_native,lazy_e2e"`) to run all passes or targeted backfills.

In [10]:
governance_report: dict[str, Any] = {}
pass4_biglake_records: list[dict[str, Any]] = []
pass5_native_records: list[dict[str, Any]] = []
pass6_ctas_native_records: list[dict[str, Any]] = []
pass7_lazy_e2e_records: list[dict[str, Any]] = []

try:
    governance_report = verify_slot_reservation_governance(
        client=bq_client,
        project_id=PROJECT_ID,
        ds_bronze_native=DS_BRONZE_NATIVE,
        reservation_name=BQ_RESERVATION_NAME,
        info_schema_region=BQ_INFO_SCHEMA_REGION,
        allow_unreserved_bq=ALLOW_UNRESERVED_BQ,
    )

    if "biglake" in SELECTED_BF_PASSES:
        pass4_biglake_records = run_medallion_pass(
            bf_session=session,
            client=bq_client,
            engine_variant="bigframes_biglake_gcs",
            storage_backend="biglake_gcs_parquet",
            bronze_ds=DS_BRONZE_BIGLAKE,
            silver_ds=DS_SILVER_BQ,
            gold_ds=DS_GOLD_BQ,
            suffix="biglake",
            use_ctas=False,
            use_daily_cache=True,
        )
    else:
        print("[SKIP] Pass 4 (bigframes_biglake_gcs) skipped via BF_PASS_FILTER.")

    if "native" in SELECTED_BF_PASSES:
        pass5_native_records = run_medallion_pass(
            bf_session=session,
            client=bq_client,
            engine_variant="bigframes_native_bq",
            storage_backend="bq_native_capacitor",
            bronze_ds=DS_BRONZE_NATIVE,
            silver_ds=DS_SILVER_BQ,
            gold_ds=DS_GOLD_BQ,
            suffix="native",
            use_ctas=False,
            use_daily_cache=True,
        )
    else:
        print("[SKIP] Pass 5 (bigframes_native_bq) skipped via BF_PASS_FILTER.")

    if "ctas_native" in SELECTED_BF_PASSES:
        pass6_ctas_native_records = run_medallion_pass(
            bf_session=session,
            client=bq_client,
            engine_variant="bigframes_ctas_native_bq",
            storage_backend="bq_native_capacitor",
            bronze_ds=DS_BRONZE_NATIVE,
            silver_ds=DS_SILVER_BQ,
            gold_ds=DS_GOLD_BQ,
            suffix="ctas_native",
            use_ctas=True,
            use_daily_cache=False,
        )
    else:
        print("[SKIP] Pass 6 (bigframes_ctas_native_bq) skipped via BF_PASS_FILTER.")

    if "lazy_e2e" in SELECTED_BF_PASSES:
        pass7_lazy_e2e_records = run_lazy_e2e_pass(
            bf_session=session,
            client=bq_client,
            engine_variant="bigframes_lazy_e2e_bq",
            storage_backend="bq_native_capacitor",
            bronze_ds=DS_BRONZE_NATIVE,
            gold_ds=DS_GOLD_BQ,
            suffix="lazy_e2e",
        )
    else:
        print("[SKIP] Pass 7 (bigframes_lazy_e2e_bq) skipped via BF_PASS_FILTER.")
finally:
    bpd.close_session()
    print("Closed BigFrames session cleanly (_anonymous_* session artifacts purged).")

[Governance Probe] job_id=8225c9b5-ea00-45f8-9691-8f10c6bd4d72 | reservation_id=<your-gcp-project-id>:US.bench-elt-slots | edition=ENTERPRISE | info_bytes_billed=10485760 | total_slot_ms=7
[PASS] Probe query executed inside assigned slot reservation '<your-gcp-project-id>:US.bench-elt-slots' (edition=ENTERPRISE; billed by slot-hours, $0 On-Demand byte charges).


[WARN] Caller succeeded with SET @@reservation='none' (job_id=8c663a2d-e50a-4a0a-b66e-46d0d283a1ee, reservation_id=none). This occurs when executing as a Sandbox roles/owner admin holding bigquery.reservations.* override permissions; production workloads must use roles/bigquery.jobUser (without reservation admin) to make slot caps non-bypassable.

=== Starting bigframes_biglake_gcs (biglake_gcs_parquet | to_gbq(replace)) | bronze=bench_bronze_biglake -> silver=bench_silver_bq, gold=bench_gold_bq (suffix=_biglake) ===


[bigframes_biglake_gcs | stage1_bronze_to_silver] wall_clock=163.16s | rows=1,014,392,792 | sample_frac=1.000000 | checksum=1,841,729,847,639.57 | slot_ms=57,689,363 | bytes=241,836,383,594 | res=<your-gcp-project-id>:US.bench-elt-slots | bi=NONE


[bigframes_biglake_gcs | stage2a_silver_vectorized] wall_clock=159.98s | rows=1,014,392,792 | sample_frac=1.000000 | checksum=1,841,729,847,639.56 | slot_ms=60,831,508 | bytes=296,509,547,782 | res=<your-gcp-project-id>:US.bench-elt-slots | bi=NONE


[bigframes_biglake_gcs | stage2b_silver_udf] wall_clock=22.14s | rows=50,660,213 | sample_frac=0.049941 | checksum=92,004,035,538.14 | slot_ms=3,628,600 | bytes=129,058,905,464 | res=<your-gcp-project-id>:US.bench-elt-slots | bi=NONE


[bigframes_biglake_gcs | stage3_silver_to_gold] wall_clock=49.76s | rows=28,068 | sample_frac=1.000000 | checksum=1,837,183,563,044.75 | slot_ms=12,686,102 | bytes=93,087,002,398 | res=<your-gcp-project-id>:US.bench-elt-slots | bi=NONE

=== Starting bigframes_native_bq (bq_native_capacitor | to_gbq(replace)) | bronze=bench_bronze_native -> silver=bench_silver_bq, gold=bench_gold_bq (suffix=_native) ===


... (6 outputs left out)

[bigframes_ctas_native_bq | stage2b_silver_udf] wall_clock=8.34s | rows=50,660,213 | sample_frac=0.049941 | checksum=92,004,035,538.14 | slot_ms=1,358,261 | bytes=120,241,354,450 | res=<your-gcp-project-id>:US.bench-elt-slots | bi=NONE


[bigframes_ctas_native_bq | stage3_silver_to_gold] wall_clock=61.04s | rows=28,068 | sample_frac=1.000000 | checksum=1,837,183,563,044.75 | slot_ms=11,568,048 | bytes=0 | res=<your-gcp-project-id>:US.bench-elt-slots | bi=NONE

=== Starting bigframes_lazy_e2e_bq (bq_native_capacitor | lazy E2E df.sql CTAS) | bronze=bench_bronze_native -> gold=bench_gold_bq (suffix=_lazy_e2e) ===


[bigframes_lazy_e2e_bq | stage_e2e_bronze_to_gold] wall_clock=123.55s | rows=28,068 | sample_frac=1.000000 | checksum=1,837,183,563,044.75 | slot_ms=45,273,692 | bytes=92,259,389,576 | res=<your-gcp-project-id>:US.bench-elt-slots | bi=NONE


Closed BigFrames session cleanly (_anonymous_* session artifacts purged).


### End-of-Run BigFrames Storage, CTAS & Lazy E2E Parity Summary

Validates **100% row-count equality** and **`±0.01%` `checksum_net_paid` tolerance** across all executed passes (plus existing `SCALE_PRESET` live records in `results/live_runs/benchmark_runs.jsonl` during selective `BF_PASS_FILTER` backfills), and compares BigLake vs. Native `to_gbq(replace)` vs. Native `df.sql` Clustered CTAS vs. Lazy Bronze -> Gold E2E.

In [11]:
all_records = (
    pass4_biglake_records
    + pass5_native_records
    + pass6_ctas_native_records
    + pass7_lazy_e2e_records
)
assert all_records, "Expected at least one BigFrames pass to execute."

summary_df = pd.DataFrame(
    [
        {
            "engine_variant": r["engine_variant"],
            "storage_backend": r["storage_backend"],
            "stage_id": r["stage_id"],
            "wall_clock_sec": round(float(r["wall_clock_sec"]), 3),
            "output_row_count": int(r["output_row_count"]),
            "sample_fraction": float(r["sample_fraction"]),
            "checksum_net_paid": float(r["checksum_net_paid"]),
            "bq_total_slot_ms": int(r["bq_total_slot_ms"]),
            "bq_bytes_processed": int(r["bq_bytes_processed"]),
            "est_list_cost_usd": (
                round(float(r["est_list_cost_usd"]), 6)
                if r.get("est_list_cost_usd") is not None
                else None
            ),
            "reservation_id": r["reservation_id"],
            "bi_engine_mode": r["bi_engine_mode"],
        }
        for r in all_records
    ]
)

display(summary_df)


def _load_existing_live_scale_records(path: Path, scale_preset: str) -> list[dict[str, Any]]:
    if not path.is_file():
        return []
    rows: list[dict[str, Any]] = []
    with path.open("r", encoding="utf-8") as fh:
        for line in fh:
            stripped = line.strip()
            if not stripped:
                continue
            rec = json.loads(stripped)
            if rec.get("scale_preset") == scale_preset:
                rows.append(rec)
    return rows


# Merge existing SCALE_PRESET live records with current-run records (latest per engine+stage wins).
merged_by_key: dict[tuple[str, str], dict[str, Any]] = {}
for rec in _load_existing_live_scale_records(METRICS_JSONL_PATH, SCALE_PRESET):
    merged_by_key[(str(rec["engine_variant"]), str(rec["stage_id"]))] = rec
for rec in all_records:
    merged_by_key[(str(rec["engine_variant"]), str(rec["stage_id"]))] = rec

scale_records = list(merged_by_key.values())

# 1. Fail-closed stage parity across all executed staged passes + existing SCALE_PRESET live records.
STAGED_IDS = (
    "stage1_bronze_to_silver",
    "stage2a_silver_vectorized",
    "stage2b_silver_udf",
    "stage3_silver_to_gold",
)
for stage_id in STAGED_IDS:
    stage_rows = [r for r in scale_records if r["stage_id"] == stage_id]
    if len(stage_rows) < 2:
        continue
    # Group by rounded sample_fraction so Stage 2B sampled (5%) vs full (100%) are verified cleanly.
    by_frac: dict[float, list[dict[str, Any]]] = {}
    for r in stage_rows:
        by_frac.setdefault(round(float(r.get("sample_fraction", 1.0)), 4), []).append(r)
    for frac, group in by_frac.items():
        if len(group) < 2:
            continue
        ref = group[0]
        ref_rows = int(ref["output_row_count"])
        ref_chk = float(ref["checksum_net_paid"])
        for r in group[1:]:
            cur_rows = int(r["output_row_count"])
            cur_chk = float(r["checksum_net_paid"])
            rel_diff = abs(cur_chk - ref_chk) / max(abs(ref_chk), 1.0)
            assert cur_rows == ref_rows, (
                f"{stage_id} (sample_frac={frac}) row count mismatch: "
                f"{ref['engine_variant']}={ref_rows} vs {r['engine_variant']}={cur_rows}"
            )
            assert rel_diff <= 1e-4, (
                f"{stage_id} (sample_frac={frac}) checksum mismatch > 0.01%: "
                f"{ref['engine_variant']}={ref_chk} vs {r['engine_variant']}={cur_chk} (rel_diff={rel_diff:.6f})"
            )

# 2. Final Gold KPI Mart parity across ALL staged Stage 3 rows AND Pass 7 stage_e2e_bronze_to_gold.
gold_rows = [
    {
        "label": f"{r['engine_variant']}:{r['stage_id']}",
        "output_row_count": int(r["output_row_count"]),
        "checksum_net_paid": float(r["checksum_net_paid"]),
    }
    for r in scale_records
    if r["stage_id"] in {"stage3_silver_to_gold", "stage_e2e_bronze_to_gold"}
]

if pass7_lazy_e2e_records and not any(":stage3_silver_to_gold" in g["label"] for g in gold_rows):
    for tbl_suffix in ("ctas_native", "native", "biglake"):
        cand_tbl = f"{PROJECT_ID}.{DS_GOLD_BQ}.gold_omnichannel_kpi_mart_{tbl_suffix}"
        try:
            cand_rows, cand_chk = fetch_materialized_checksum(
                client=bq_client,
                table_fqn=cand_tbl,
                checksum_col="daily_net_paid",
            )
            if cand_rows > 0:
                gold_rows.append(
                    {
                        "label": f"bq_table:gold_omnichannel_kpi_mart_{tbl_suffix}",
                        "output_row_count": cand_rows,
                        "checksum_net_paid": cand_chk,
                    }
                )
        except gapi_exceptions.NotFound:
            continue

if pass7_lazy_e2e_records:
    assert len(gold_rows) >= 2, (
        "Pass 7 (bigframes_lazy_e2e_bq) requires at least one Gold reference "
        "(stage3_silver_to_gold in current run / JSONL or gold_omnichannel_kpi_mart_* table)."
    )

if len(gold_rows) >= 2:
    gold_ref = gold_rows[0]
    for g in gold_rows[1:]:
        rel_diff = abs(g["checksum_net_paid"] - gold_ref["checksum_net_paid"]) / max(
            abs(gold_ref["checksum_net_paid"]), 1.0
        )
        assert g["output_row_count"] == gold_ref["output_row_count"], (
            f"Gold KPI Mart row mismatch: {gold_ref['label']}={gold_ref['output_row_count']} "
            f"vs {g['label']}={g['output_row_count']}"
        )
        assert rel_diff <= 1e-4, (
            f"Gold KPI Mart checksum mismatch > 0.01%: {gold_ref['label']}={gold_ref['checksum_net_paid']} "
            f"vs {g['label']}={g['checksum_net_paid']} (rel_diff={rel_diff:.6f})"
        )

# 3. Verify Stage 2A vs Stage 2B exact parity on small_1g across executed staged passes.
if SCALE_PRESET == "small_1g":
    for label, pass_recs in [
        ("biglake", pass4_biglake_records),
        ("native", pass5_native_records),
        ("ctas_native", pass6_ctas_native_records),
    ]:
        by_s = {r["stage_id"]: r for r in pass_recs}
        if "stage2a_silver_vectorized" in by_s and "stage2b_silver_udf" in by_s:
            s2a, s2b = by_s["stage2a_silver_vectorized"], by_s["stage2b_silver_udf"]
            assert s2a["output_row_count"] == s2b["output_row_count"], (
                f"{label} Stage 2A vs 2B row mismatch on small_1g"
            )
            assert (
                abs(s2a["checksum_net_paid"] - s2b["checksum_net_paid"])
                / max(abs(s2a["checksum_net_paid"]), 1.0)
                <= 1e-4
            )

print(
    f"[PASS] 100% row-count parity and <=0.01% checksum parity verified across "
    f"{len(gold_rows)} Gold checkpoints for {SCALE_PRESET}."
)

# Build BigFrames Core Relational Comparison Table (S1+S2A+S3 for staged; E2E for lazy_e2e).
BF_ORDER = (
    "bigframes_biglake_gcs",
    "bigframes_native_bq",
    "bigframes_ctas_native_bq",
    "bigframes_lazy_e2e_bq",
)
bf_rows = []
for eng in BF_ORDER:
    eng_recs = {r["stage_id"]: r for r in scale_records if r["engine_variant"] == eng}
    if eng == "bigframes_lazy_e2e_bq" and "stage_e2e_bronze_to_gold" in eng_recs:
        r = eng_recs["stage_e2e_bronze_to_gold"]
        core_sec = float(r["wall_clock_sec"])
        core_slot_ms = int(r["bq_total_slot_ms"] or 0)
        core_bytes = int(r["bq_bytes_processed"] or 0)
        core_cost = float(r.get("est_list_cost_usd") or 0.0)
        gold_chk = float(r["checksum_net_paid"])
        gold_cnt = int(r["output_row_count"])
    elif all(s in eng_recs for s in ("stage1_bronze_to_silver", "stage2a_silver_vectorized", "stage3_silver_to_gold")):
        core_stages = [
            eng_recs["stage1_bronze_to_silver"],
            eng_recs["stage2a_silver_vectorized"],
            eng_recs["stage3_silver_to_gold"],
        ]
        core_sec = sum(float(r["wall_clock_sec"]) for r in core_stages)
        core_slot_ms = sum(int(r["bq_total_slot_ms"] or 0) for r in core_stages)
        core_bytes = sum(int(r["bq_bytes_processed"] or 0) for r in core_stages)
        core_cost = sum(float(r.get("est_list_cost_usd") or 0.0) for r in core_stages)
        gold_chk = float(eng_recs["stage3_silver_to_gold"]["checksum_net_paid"])
        gold_cnt = int(eng_recs["stage3_silver_to_gold"]["output_row_count"])
    else:
        continue
    bf_rows.append(
        {
            "engine_variant": eng,
            "core_wall_clock_sec": round(core_sec, 2),
            "core_slot_seconds": round(core_slot_ms / 1000.0, 2),
            "core_bytes_gib": round(core_bytes / (1024 ** 3), 3),
            "core_est_list_usd": round(core_cost, 6),
            "gold_rows": gold_cnt,
            "gold_checksum_net_paid": round(gold_chk, 2),
        }
    )

if bf_rows:
    bf_cmp_df = pd.DataFrame(bf_rows)
    base_native = bf_cmp_df.loc[bf_cmp_df["engine_variant"] == "bigframes_native_bq"]
    if not base_native.empty:
        base_sec = float(base_native.iloc[0]["core_wall_clock_sec"])
        base_slots = float(base_native.iloc[0]["core_slot_seconds"])
        bf_cmp_df["speedup_vs_native_x"] = (
            base_sec / bf_cmp_df["core_wall_clock_sec"].clip(lower=1e-6)
        ).round(2)
        bf_cmp_df["slot_reduction_pct_vs_native"] = (
            (1.0 - bf_cmp_df["core_slot_seconds"] / max(base_slots, 1e-6)) * 100.0
        ).round(1)
    display(bf_cmp_df)

,engine_variant,storage_backend,stage_id,wall_clock_sec,output_row_count,sample_fraction,checksum_net_paid,bq_total_slot_ms,bq_bytes_processed,est_list_cost_usd,reservation_id,bi_engine_mode
0,bigframes_biglake_gcs,biglake_gcs_parquet,stage1_bronze_to_silver,163.163,1014392792,1.000000,1.841730e+12,57689363,241836383594,0.961489,:US.bench-elt-slots,NONE
1,bigframes_biglake_gcs,biglake_gcs_parquet,stage2a_silver_vectorized,159.977,1014392792,1.000000,1.841730e+12,60831508,296509547782,1.013858,:US.bench-elt-slots,NONE
2,bigframes_biglake_gcs,biglake_gcs_parquet,stage2b_silver_udf,22.136,50660213,0.049941,9.200404e+10,3628600,129058905464,0.060477,:US.bench-elt-slots,NONE
3,bigframes_biglake_gcs,biglake_gcs_parquet,stage3_silver_to_gold,49.761,28068,1.000000,1.837184e+12,12686102,93087002398,0.211435,:US.bench-elt-slots,NONE
4,bigframes_native_bq,bq_native_capacitor,stage1_bronze_to_silver,157.442,1014392792,1.000000,1.841730e+12,57419523,241836383594,0.956992,:US.bench-elt-slots,NONE
5,bigframes_native_bq,bq_native_capacitor,stage2a_silver_vectorized,158.804,1014392792,1.000000,1.841730e+12,54964285,120241354450,0.916071,:US.bench-elt-slots,NONE
6,bigframes_native_bq,bq_native_capacitor,stage2b_silver_udf,21.294,50660213,0.049941,9.200404e+10,2231396,120241354450,0.037190,:US.bench-elt-slots,NONE
7,bigframes_native_bq,bq_native_capacitor,stage3_silver_to_gold,47.079,28068,1.000000,1.837184e+12,12297718,93081742988,0.204962,:US.bench-elt-slots,NONE
8,bigframes_ctas_native_bq,bq_native_capacitor,stage1_bronze_to_silver,79.686,1014392792,1.000000,1.841730e+12,27605826,91222377008,0.460097,:US.bench-elt-slots,NONE
9,bigframes_ctas_native_bq,bq_native_capacitor,stage2a_silver_vectorized,63.642,1014392792,1.000000,1.841730e+12,19349749,120241354450,0.322496,:US.bench-elt-slots,NONE


[PASS] 100% row-count parity and <=0.01% checksum parity verified across 7 Gold checkpoints for large_500g.


,engine_variant,core_wall_clock_sec,core_slot_seconds,core_bytes_gib,core_est_list_usd,gold_rows,gold_checksum_net_paid,speedup_vs_native_x,slot_reduction_pct_vs_native
0,bigframes_biglake_gcs,372.90,131206.97,588.068,2.186782,28068,1.837184e+12,0.97,-5.2
1,bigframes_native_bq,363.33,124681.53,423.900,2.078025,28068,1.837184e+12,1.00,0.0
2,bigframes_ctas_native_bq,204.37,58523.62,196.941,0.975394,28068,1.837184e+12,1.78,53.1
3,bigframes_lazy_e2e_bq,123.55,45273.69,85.923,0.754562,28068,1.837184e+12,2.94,63.7
